In [1]:
from datasets import load_dataset
import json
import numpy as np
import random


In [2]:
system_prompt = """You are an event extraction system. 
Your task is to assign the correct event type and argument roles to a given trigger and its arguments.
You will be provided with:
- an input text
- a trigger span
- a list of argument spans

IMPORTANT:Output ONLY valid JSON. No explanations, no markdown, no extra text.
Output Format (JSON only, no markdown):

{"events": [[<trigger span>, <event type>, [[<argument span>, <semantic role>], [<argument span 2>, <semantic role 2>]]], [<trigger span 2>, <event type 2>, []]]}

- If no events are detected, return: {"events": []}"""

user_prompt = """
Input text:
{text}

Trigger:
{trigger}

Arguments:
{arguments}

Task:
- Assign the most appropriate event type to the trigger.
- Assign the most appropriate role to each argument.

Rules: Do NOT change the trigger or argument spans.
"""

In [3]:
t_system_prompt = """You are an event extraction system. 
Your task is to assign the correct event type and argument roles to a given trigger and its arguments.
You will be provided with:
- an input text
- a trigger span
- a list of argument spans
- a predefined set of event types
- a predefined set of semantic roles

IMPORTANT:Output ONLY valid JSON. No explanations, no markdown, no extra text.
Output Format (JSON only, no markdown):
{"events": [[<trigger span>, <event type>, [[<argument span>, <semantic role>], [<argument span 2>, <semantic role 2>]]]]}

- If no events are detected, return: {"events": []}"""

t_user_prompt = """
Input text:
{text}

Trigger:
{trigger}

Arguments:
{arguments}

Available Event Types:
{event_types}

Available Argument Roles:
{roles}

Task:
- Assign the most appropriate event type to the trigger.
- Assign the most appropriate role to each argument.

Rules:
- Use ONLY labels from the provided lists.
- Do NOT change the trigger or argument spans.

Here is a reference response to the input sentence:
{result}
You may include additional valid triggers not in the reference. Now provide your own extraction, including the thinking process::
"""

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-4B', padding_side="right")

In [5]:
def extract_unique_labels(dataset):
    event_types = set()
    argument_roles = set()
    
    for sample in dataset:
        events = sample.get('events', [])
        
        for event in events:
            if 'type' in event:
                event_types.add(event['type'])
                
            mentions = event.get('mention', [])
            for mention in mentions:
                arguments = mention.get('arguments', [])
                
                for arg in arguments:
                    if 'role' in arg:
                        argument_roles.add(arg['role'])
                        
    return list(event_types), list(argument_roles)

In [6]:
def process_data(raw_data, buffer, label2id, is_test=False, tasks=[], eval_tasks=[]):
    full_len = []
    sent_total = 0
    data = []
    prompt_len = []
    response_len = []
    none_data = []

    temp_buffer = {}
    b_sent_id_map = {}

    event_types, argument_roles = extract_unique_labels(raw_data)

    for idx, sample in enumerate(raw_data):
        sent_id_to_sentence = {i: content['sentence'] for i, content in enumerate(sample["content"])}
        sent_id_set = set(sent_id_to_sentence.keys())
        sent_total += len(sent_id_to_sentence)
        sent_to_existing_events = {}

        for event in sample.get("events", []):
            if event.get("type_id", -1) == -1: continue
            event_type = event.get("type")
            description = event.get("description", "")
            
            if label2id[event_type] not in tasks:
                if not (is_test and label2id[event_type] in eval_tasks):
                    continue

            if event_type not in temp_buffer:
                temp_buffer[event_type] = []
            
            for mention in event.get("mention", []):
                sent_id = mention.get("sent_id")
                             
                if sent_id not in sent_to_existing_events:
                    sent_to_existing_events[sent_id] = []
                args = [[arg["text"], arg["role"]] for arg in mention.get("arguments", [])]
                event_info = [mention.get("trigger_word"), event_type, args, description]
                
                sent_to_existing_events[sent_id].append(event_info)

                if len(temp_buffer[event_type]) < 10:
                    b_sent_id_map[f"{idx}_{sent_id}"] = event_type

        for sent_id, events in sent_to_existing_events.items():
            sent_txt = sent_id_to_sentence[sent_id]
            sent_id_set.remove(sent_id)

            triggers = [item[0] for item in events]
            args = [arg[0] for item in events for arg in item[2]]

            response = json.dumps({"events": events})

            data.append({"system_prompt": system_prompt, 
                         "user_prompt": user_prompt.format(text=sent_txt, trigger=triggers, arguments=args, 
                                                           event_types=event_types, roles=argument_roles, result=response), 
                         "response": response, 
                         "t_system_prompt": t_system_prompt,
                         "t_user_prompt": t_user_prompt.format(text=sent_txt, trigger=triggers, arguments=args, 
                                                               event_types=event_types, roles=argument_roles, result=response), 
                         })

            if f"{idx}_{sent_id}" in b_sent_id_map:
                temp_buffer[b_sent_id_map[f"{idx}_{sent_id}"]].append(data[-1])


        for sent_id in sent_id_set:
            sent_txt = sent_id_to_sentence[sent_id]

            response = json.dumps({"events": []})

            none_data.append({"system_prompt": system_prompt, 
                              "user_prompt": user_prompt.format(text=sent_txt, trigger=[], arguments=[], 
                                                                event_types=event_types, roles=argument_roles), 
                              "response": response, 
                              "t_system_prompt": t_system_prompt,
                              "t_user_prompt": t_user_prompt.format(text=sent_txt, trigger=[], arguments=[], 
                                                                    event_types=event_types, roles=argument_roles, result=response), 
                              })


    if is_test:
        # data.extend(none_data)
        pass
    else:
        # data.extend(random.sample(list(none_data), min(len(none_data), len(data) // 100)))
        data.extend(buffer)
        
        for k, v in temp_buffer.items():
            buffer.extend(v)

    # for sample in data:
    #     prompt = tokenizer.apply_chat_template(
    #             [{"role": "system", "content": sample['system_prompt']},
    #             {"role": "user", "content": sample['user_prompt']}],
    #             add_generation_prompt=True,
    #             tokenize=False 
    #         )
    #     full = prompt + sample['response'] + tokenizer.eos_token
        
    #     prompt_tokens = tokenizer.encode(prompt, add_special_tokens=False)
    #     full_tokens = tokenizer.encode(full, add_special_tokens=False)
    #     response_tokens = full_tokens[len(prompt_tokens):]

    #     full_len.append(len(full_tokens))
    #     prompt_len.append(len(prompt_tokens))
    #     response_len.append(len(response_tokens))
    
    return data, full_len, prompt_len, response_len

In [7]:
import os

def save(data, data_name, tasks, label2id, buffer, eval_tasks): 
    os.makedirs(f"data/{data_name}", exist_ok=True)
    train, _, _, _ = process_data(data["train"], buffer=buffer, label2id=label2id, tasks=tasks)
    with open(f"data/{data_name}/train.jsonl", "w", encoding="utf-8") as f:
        for item in train:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    dev, _, prompt_len, _ = process_data(data["validation"], [], label2id=label2id, is_test=True, tasks=tasks, eval_tasks=eval_tasks)
    with open(f"data/{data_name}/dev.jsonl", "w", encoding="utf-8") as f:
        for item in dev:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    test, _, prompt_len, _ = process_data(data["test"], [], label2id=label2id, is_test=True, tasks=tasks, eval_tasks=eval_tasks)
    with open(f"data/{data_name}/test.jsonl", "w", encoding="utf-8") as f:
        for item in test:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    return train, dev, test

In [8]:
ace_gen = load_dataset("datht/ace-short-generated-dataset")
geneva_gen = load_dataset("datht/geneva-short-generated-dataset")
maven_gen = load_dataset("datht/maven-short-generated-dataset")
rams_gen = load_dataset("datht/rams-short-generated-dataset")


In [44]:
train, full_len, prompt_len, response_len = process_data(rams_gen["train"], buffer=[], label2id=[], tasks=[])


In [48]:
len(train)

7329

In [41]:
np.median(np.array(prompt_len))

np.float64(261.0)

In [47]:
np.sum(np.array(full_len) > 460)

np.int64(43)

In [55]:


with open("data/ext-data/ace/label2id.json", "r", encoding="utf-8") as f:
    label2id = json.load(f)

with open("data/ext-data/ace/streams.json", "r", encoding="utf-8") as f:
    streams = json.load(f)

buffer = []
eval_tasks = []
for i, tasks in enumerate(streams):
    eval_tasks.extend(tasks)
    save(ace_gen, f"ace_v3/{i}", tasks, label2id, buffer, eval_tasks)

In [56]:


with open("data/ext-data/geneva/label2id.json", "r", encoding="utf-8") as f:
    label2id = json.load(f)

with open("data/ext-data/geneva/streams.json", "r", encoding="utf-8") as f:
    streams = json.load(f)

buffer = []
eval_tasks = []
for i, tasks in enumerate(streams):
    eval_tasks.extend(tasks)
    save(geneva_gen, f"geneva_v3/{i}", tasks, label2id, buffer, eval_tasks)

In [9]:

with open("data/ext-data/maven/label2id.json", "r", encoding="utf-8") as f:
    label2id = json.load(f)

with open("data/ext-data/maven/streams.json", "r", encoding="utf-8") as f:
    streams = json.load(f)

buffer = []
eval_tasks = []
for i, tasks in enumerate(streams):
    eval_tasks.extend(tasks)
    save(maven_gen, f"maven_v3/{i}", tasks, label2id, buffer, eval_tasks)

In [10]:

with open("data/ext-data/rams/label2id.json", "r", encoding="utf-8") as f:
    label2id = json.load(f)

with open("data/ext-data/rams/streams.json", "r", encoding="utf-8") as f:
    streams = json.load(f)

buffer = []
eval_tasks = []
for i, tasks in enumerate(streams):
    eval_tasks.extend(tasks)
    save(rams_gen, f"rams_v3/{i}", tasks, label2id, buffer, eval_tasks)